In [37]:
from itertools import batched
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
import json
import hashlib
import numpy as np
from tqdm import tqdm
import random

In [17]:
def record_text(record):
    if record.get("turns", None) is None:
        return record["text"]
    else:
        return '\n'.join([i["text"] for i in record["turns"]])
def split_of(record, val_share, test_share):
    val_share = val_share / 100
    test_share = test_share / 100
    string = "|".join([raw["source"], raw["source_split"], raw["source_id"]])
    hash_str = hashlib.sha256(string.encode()).hexdigest()
    hash_int = int(hash_str, 16) % 1000
    test_tr = 1000 - 1000 * test_share
    train_tr = test_tr - 1000 * val_share
    if hash_int < train_tr:
        ans = "train"
    elif hash_int < test_tr:
        ans = "valid"
    else:
        ans = "test"
    return ans
def read_jsonl(path):
    with open(path, encoding="UTF-8") as data:
        for obj in data:
            if is_empty(obj):
                continue
            yield json.loads(obj)
is_empty = lambda text: text is None or text.strip() == ""

In [8]:
def generate_for_tokenizer(pathsft, pathpretrain, mode="train", max_docs=50_000):
    for raw in read_jsonl(pathsft):
        for phrase in raw["turns"]:
            yield phrase["text"]
    counter=0
    for raw in read_jsonl(pathpretrain):
        if counter >= max_docs:
            break
        if split_of(raw, 0.5, 0.5) == mode:
            counter += 1
            yield record_text(raw)
def train_tokenizer(V, max_docs):
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tok.decoder = decoders.ByteLevel()
    
    trainer = trainers.BpeTrainer(
    vocab_size=V,
    special_tokens=["<pad>", "<mask>", "<user>", "<bot>", "<eos>"] + [f"<reserved_{i}>" for i in range(16)],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    min_frequency=2,
    )

    tok.train_from_iterator(generate_for_tokenizer(TRAIN_SFT, PRETRAIN, "train", max_docs), trainer=trainer)
    return tok
def chars_per_token(tok, texts):
    encoded = tok.encode_batch(texts)
    n_encoded, len_raw = 0, 0
    for tokens_per_obj in encoded:
        n_encoded += len(tokens_per_obj.ids)
    for raw in texts:
        len_raw += len(raw)
    return len_raw / n_encoded  

In [10]:
TRAIN_SFT = "/kaggle/input/datasets/absolute1000/training-llm/TRAIN_SFT.jsonl"
VALID_SFT = "/kaggle/input/datasets/absolute1000/training-llm/VALID_SFT.jsonl"
TEST_SFT = "/kaggle/input/datasets/absolute1000/training-llm/TEST_SFT.jsonl"
PRETRAIN = "/kaggle/input/datasets/absolute1000/training-llm/pretrain_normalized.jsonl"

In [22]:
texts_val_sft = [phrase["text"] for _ in read_jsonl(VALID_SFT) for phrase in _["turns"]]
texts_val_pretrain = []
counter = 0
for raw in read_jsonl(PRETRAIN):
    if counter >= 2000:
        break
    if split_of(raw, 0.5, 0.5) == "valid":
        counter += 1
        texts_val_pretrain.append(record_text(raw))

In [23]:
for v in (8000, 12000, 16000):
    tokenizer = train_tokenizer(v, 50_000)
    sft_stats = chars_per_token(tokenizer, texts_val_sft)
    pretrain_stats = chars_per_token(tokenizer, texts_val_pretrain)
    print(f"TOKENIZER dict_size {v} | CPT_sft {sft_stats} | CPT_pretrain {pretrain_stats}")




TOKENIZER dict_size 8000 | CPT_sft 3.6662281278779107 | CPT_pretrain 3.9080426589688075



TOKENIZER dict_size 12000 | CPT_sft 3.803018723729461 | CPT_pretrain 4.15235935615792



TOKENIZER dict_size 16000 | CPT_sft 3.8823332729628444 | CPT_pretrain 4.309009890485565


In [24]:
tokenizer = train_tokenizer(12_288, 500_000)

In [25]:
bugssft, pretrainsft = list(), list()
stf_pred = tokenizer.encode_batch(texts_val_sft)
pretrain_pred = tokenizer.encode_batch(texts_val_pretrain)
for i in range(len(stf_pred)):
    pred = tokenizer.decode(stf_pred[i].ids)
    if pred != texts_val_sft[i]:
        bugssft.append((repr(pred), repr(texts_val_sft[i])))
for i in range(len(pretrain_pred)):
    pred = tokenizer.decode(pretrain_pred[i].ids)
    if pred != texts_val_pretrain[i]:
        pretrainsft.append((repr(pred), repr(texts_val_pretrain[i])))
print(len(bugssft), len(pretrainsft))
print(bugssft[:20])
print(pretrainsft[:10])

0 0
[]
[]


In [27]:
tokenizer.save("/kaggle/working/tokenizer.json")

In [32]:
pretrain_train = "/kaggle/working/pretrain_train.bin"
pretrain_valid = "/kaggle/working/pretrain_valid.bin"
eos = tokenizer.token_to_id("<eos>")
train_tokens, val_tokens = 0, 0
with (
    open(pretrain_train, "wb") as pretrain_trainf,
    open(pretrain_valid, "wb") as pretrain_validf
):
    to_train, to_val = [], []
    for raw in tqdm(read_jsonl(PRETRAIN)):
        split = split_of(raw, 0.5, 0.5)
        text = record_text(raw)
        if split == "train":
            to_train.append(text)
        elif split == "valid":
            to_val.append(text)

        if len(to_train) == 10_000:
            encoded = tokenizer.encode_batch(to_train)
            for obj in encoded:
                obj = obj.ids
                train_tokens += len(obj)
                obj = np.array(obj + [eos], dtype=np.uint16)
                obj.tofile(pretrain_trainf)
                to_train = []
        if len(to_val) == 10_000:
            encoded = tokenizer.encode_batch(to_val)
            for obj in encoded:
                obj = obj.ids
                val_tokens += len(obj)
                obj = np.array(obj + [eos], dtype=np.uint16)
                obj.tofile(pretrain_validf) 
                to_val = []
    encoded = tokenizer.encode_batch(to_train)
    for obj in encoded:
        obj = obj.ids
        train_tokens += len(obj)
        obj = np.array(obj + [eos], dtype=np.uint16)
        obj.tofile(pretrain_trainf)
        to_train = []
    encoded = tokenizer.encode_batch(to_val)
    for obj in encoded:
        obj = obj.ids
        val_tokens += len(obj)
        obj = np.array(obj + [eos], dtype=np.uint16)
        obj.tofile(pretrain_validf) 
        to_val = []
print(train_tokens, val_tokens)                        

2163545it [1:20:23, 448.59it/s] 


2457334417 12062582


In [34]:
def pack_sft(src_path, split_name):
    with (
        open(f"/kaggle/working/sft_{split_name}_tokens.bin", "wb") as tks,
        open(f"/kaggle/working/sft_{split_name}_mask.bin", "wb") as masks
    ):
        offsets = []
        position = 0
        all_tokens, tokens_masked = 0, 0
        for raw in read_jsonl(src_path):
            offsets.append(position)
            dialogue_len = 0
            whole_dialogue_encoded = tokenizer.encode_batch([phrase["text"] for phrase in raw["turns"]])
            for phrase in zip(raw["turns"], whole_dialogue_encoded):
                if phrase[0]["role"] == "user":
                    left = tokenizer.encode_batch(["<user>"])
                    final_phrase_tokens = np.array(left[0].ids + phrase[1].ids, dtype=np.uint16)
                    mask = np.zeros(final_phrase_tokens.shape, dtype=np.uint8)
                    all_tokens += len(final_phrase_tokens)
                else:
                    left, right = tokenizer.encode_batch(["<bot>", "<eos>"])
                    final_phrase_tokens = np.array(left.ids + phrase[1].ids + right.ids, dtype=np.uint16)
                    mask = np.zeros(final_phrase_tokens.shape, dtype=np.uint8)
                    mask[1:] = 1
                    all_tokens += len(final_phrase_tokens)
                    tokens_masked += len(mask) - 1
                final_phrase_tokens.tofile(tks)
                mask.tofile(masks)
                dialogue_len += len(final_phrase_tokens)
            position += dialogue_len
        offsets.append(position)
        np.save(f"/kaggle/working/sft_{split_name}_offsets.npy", np.array(offsets, dtype=np.int64))
    print(all_tokens, tokens_masked)
pack_sft(TRAIN_SFT, "train")
pack_sft(VALID_SFT, "valid")
pack_sft(TEST_SFT, "test")

5286897 2439582
82232 37799
85692 39377


In [38]:
PATH = "/kaggle/working"
user_id = tokenizer.token_to_id("<user>")
bot_id = tokenizer.token_to_id("<bot>")
eos_id = tokenizer.token_to_id("<eos>")
V = tokenizer.get_vocab_size()

def load_sft(split):
    t = np.fromfile(f"{PATH}/sft_{split}_tokens.bin", dtype=np.uint16)
    m = np.fromfile(f"{PATH}/sft_{split}_mask.bin", dtype=np.uint8)
    o = np.load(f"{PATH}/sft_{split}_offsets.npy")
    return t, m, o

print("=" * 100)
print("SHAPE CHECKS")
for split in ("train", "valid", "test"):
    t, m, o = load_sft(split)
    checks = {
        "len tokens == len mask": len(t) == len(m),
        "last offset == len tokens": o[-1] == len(t),
        "offsets strictly increase": bool((np.diff(o) > 0).all()),
        "mask only 0/1": set(np.unique(m).tolist()) <= {0, 1},
        "token ids < V": int(t.max()) < V,
        "eos in mask == bot turns": int((t[m == 1] == eos_id).sum()) == int((t == bot_id).sum()),
        "no special inside mask except eos": int(np.isin(t[m == 1], [user_id, bot_id]).sum()) == 0,
    }
    print(f"{split}: dialogs={len(o) - 1}, tokens={len(t)}")
    for name, ok in checks.items():
        print(f"   {'OK  ' if ok else 'FAIL'} {name}")

print("=" * 100)
print("DIALOG CHECKS (train)")
t, m, o = load_sft("train")
random.seed(42)
for k in [0] + random.sample(range(len(o) - 1), 2):
    seg_t = t[o[k]:o[k + 1]]
    seg_m = m[o[k]:o[k + 1]]
    print(f"--- dialog {k} ---")
    print("FULL:")
    print(tokenizer.decode(seg_t.tolist(), skip_special_tokens=False))
    print("MASK == 1 ONLY:")
    print(tokenizer.decode(seg_t[seg_m == 1].tolist(), skip_special_tokens=False))
    print()

print("=" * 100)
print("PRETRAIN CHECKS")
for split in ("train", "valid"):
    p = np.memmap(f"{PATH}/pretrain_{split}.bin", dtype=np.uint16, mode="r")
    print(f"{split}: tokens={len(p)}, eos={int((p[:10_000_000] == eos_id).sum())} in first 10M, max id ok={int(p[:10_000_000].max()) < V}")
p = np.memmap(f"{PATH}/pretrain_train.bin", dtype=np.uint16, mode="r")
print(tokenizer.decode(p[:300].tolist(), skip_special_tokens=False))

SHAPE CHECKS
train: dialogs=43732, tokens=5286897
   OK   len tokens == len mask
   OK   last offset == len tokens
   OK   offsets strictly increase
   OK   mask only 0/1
   OK   token ids < V
   OK   eos in mask == bot turns
   OK   no special inside mask except eos
valid: dialogs=692, tokens=82232
   OK   len tokens == len mask
   OK   last offset == len tokens
   OK   offsets strictly increase
   OK   mask only 0/1
   OK   token ids < V
   OK   eos in mask == bot turns
   OK   no special inside mask except eos
test: dialogs=704, tokens=85692
   OK   len tokens == len mask
   OK   last offset == len tokens
   OK   offsets strictly increase
   OK   mask only 0/1
   OK   token ids < V
   OK   eos in mask == bot turns
   OK   no special inside mask except eos
DIALOG CHECKS (train)
--- dialog 0 ---
FULL:
<user>Write a python function to find the first repeated character in a given string.<bot>```python
def first_repeated_char(str1):
  for index,c in enumerate(str1):
    if str1[:index+1]